#Project Interim Report (CS-4403)

#Team: Fady Elgohary 3762991 & Darin Thomson 3776723

In [7]:
import pandas as pd
import math
import numpy as np

In [16]:
df = pd.read_csv("/content/Data.csv")

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112 entries, 0 to 111
Data columns (total 68 columns):
 #   Column                                                                       Non-Null Count  Dtype  
---  ------                                                                       --------------  -----  
 0   Unnamed: 0                                                                   112 non-null    int64  
 1   Author and Year                                                              112 non-null    object 
 2   Percent male (%)                                                             112 non-null    int64  
 3   Race with largest number                                                     112 non-null    object 
 4   Age (years)                                                                  112 non-null    float64
 5   Height (cm)                                                                  110 non-null    float64
 6   Weight (kg)                               

In [18]:
null_data = df.isnull().sum()
print (null_data[null_data > 0])

Height (cm)                                                                     2
Weight (kg)                                                                     2
Protein intake before intervention (g/kg/day)                                  32
Protein intake before intervention (g/day)                                     32
Protein intake during intervention (not include supplementation) (g/kg/day)    68
Protein intake during intervention (not include supplementation) (g/day)       68
Protein intake during intervention (include supplementation) (g/kg/day)        45
Protein intake during intervention (include supplementation) (g/day)           43
Energy intake before intervention (g/kg/day)                                   32
Energy intake before intervention (g/day)                                      32
Energy intake during intervention (not include supplementation) (g/kg/day)     72
Energy intake during intervention (not include supplementation) (g/day)        72
Energy intake du

## Step 1: Data Type Fixes

After inspecting `df.info()`, two columns are stored as `object` but should be numeric:
- **`Control Total`** → should be `int64` (mirrors `Experimental Total`)
- **`95CI Upper`** → should be `float64` (mirrors `95CI Lower`)

All other columns already carry the correct dtype.

In [19]:
# ── Identify columns with wrong datatypes
print("Before fix:")
print(df[['Control Total', '95CI Upper']].dtypes)
print()

# convert Control Total column into an int
df['Control Total'] = pd.to_numeric(df['Control Total'], errors='coerce').astype('Int64')

# convert 95CI Upper to float
df['95CI Upper'] = pd.to_numeric(df['95CI Upper'], errors='coerce')

print("After fix:")
print(df[['Control Total', '95CI Upper']].dtypes)

Before fix:
Control Total    object
95CI Upper       object
dtype: object

After fix:
Control Total      Int64
95CI Upper       float64
dtype: object


## Step 2: Drop Irrelevant Columns

Two categories of columns are dropped:

1. **`Unnamed: 0`** — a duplicate row index with no predictive value.
2. **Columns with >50 % missing values** (out of 112 rows) — too sparse to be useful in modeling:
   | Column | Nulls | % Missing |
   |---|---|---|
   | Protein intake during intervention *(not include suppl.)* (g/kg/day) | 68 | 60.7 % |
   | Protein intake during intervention *(not include suppl.)* (g/day) | 68 | 60.7 % |
   | Energy intake during intervention *(not include suppl.)* (g/kg/day) | 72 | 64.3 % |
   | Energy intake during intervention *(not include suppl.)* (g/day) | 72 | 64.3 % |

In [20]:
#  Drop pure index column
df.drop(columns=['Unnamed: 0'], inplace=True)

#  Drop columns with more than 50% missing values
threshold = 0.50  # drop if more than 50% of values are null
cols_before = df.shape[1]

missing_frac = df.isnull().mean()          # fraction of nulls per column
cols_to_drop = missing_frac[missing_frac > threshold].index.tolist()

print(f"Columns dropped for exceeding {threshold*100:.0f}% missing threshold:")
for c in cols_to_drop:
    print(f"  '{c}'  ({df[c].isnull().sum()} / {len(df)} nulls = {missing_frac[c]*100:.1f}%)")

df.drop(columns=cols_to_drop, inplace=True)


Columns dropped for exceeding 50% missing threshold:
  'Protein intake during intervention (not include supplementation) (g/kg/day)'  (68 / 112 nulls = 60.7%)
  'Protein intake during intervention (not include supplementation) (g/day)'  (68 / 112 nulls = 60.7%)
  'Energy intake during intervention (not include supplementation) (g/kg/day)'  (72 / 112 nulls = 64.3%)
  'Energy intake during intervention (not include supplementation) (g/day)'  (72 / 112 nulls = 64.3%)


## Step 3: Fill Null Values

Imputation strategy:
- **Numeric columns** → filled with the column **median**  
  *(median is preferred over mean because biomedical measurements such as protein/energy intake are often right-skewed and contain outliers)*
- **Categorical / object columns** → filled with the column **mode** (most frequent value)

In [24]:
# Separate numeric and categorical columns
numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
object_cols  = df.select_dtypes(include=['object']).columns.tolist()

# Fill numeric nulls with median
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val, inplace=True)

# Verify no nulls remain
remaining_nulls = df.isnull().sum().sum()
print(f"\nTotal remaining null values: {remaining_nulls}")


Total remaining null values: 0


## Final Check: Clean Dataset Summary

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112 entries, 0 to 111
Data columns (total 63 columns):
 #   Column                                                                   Non-Null Count  Dtype  
---  ------                                                                   --------------  -----  
 0   Author and Year                                                          112 non-null    object 
 1   Percent male (%)                                                         112 non-null    int64  
 2   Race with largest number                                                 112 non-null    object 
 3   Age (years)                                                              112 non-null    float64
 4   Height (cm)                                                              112 non-null    float64
 5   Weight (kg)                                                              112 non-null    float64
 6   BMI (kg/m2)                                                              1

In [26]:
df.isnull().sum()

,0
Author and Year,0
Percent male (%),0
Race with largest number,0
Age (years),0
Height (cm),0
...,...
Protein intake (g/kg/day),0
LBM change (kg),0
LBM change per week (kg/wk),0
Relative LBM change,0
